In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [3]:
with open("blood_work.txt", "r") as f:
 blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


In [4]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text
print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== STAGE 1: EXTRACTED VALUES ===
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | Status: NORMAL | Reference: 10–40
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2


In [6]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

import time

for attempt in range(3):
    try:
        diet_response = llm.invoke(diet_prompt)

        print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
        print(diet_response.text)
        break

    except Exception as e:
        if attempt < 2:
            print(f"Gemini unavailable. Retrying... ({attempt + 1}/3)")
            time.sleep(5)
        else:
            print("Gemini is currently unavailable. Please try again later.")
            raise



=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
### Health Summary
Your overall blood counts, blood sugar regulation, kidney function, and liver health are all in excellent, normal ranges. However, your lipid profile shows mixed dyslipidemia, meaning your blood fat levels are out of balance. Specifically, your total cholesterol, triglycerides, and "bad" LDL cholesterol are high, while your protective "good" HDL cholesterol is low. This pattern can lead to plaque buildup in your arteries over time, increasing cardiovascular risk. Fortunately, these lipid levels can be significantly improved through targeted changes in your daily diet and lifestyle.

***

### Indian Diet Plan

#### (1) Foods to avoid
* **Deep-fried snacks and street food:** Samosas, pakoras, kachoris, puris, bhaturas, and commercial namkeens/farsan.
* **Trans fats and saturated fats:** Vanaspati (dalda), margarine, palm oil, excess butter, and excessive use of desi ghee (limit to 1 teaspoon per day).
* **Refined carbohydrate